# aeon-qc exploration

Interactive cells for testing QC metrics against benchmark datasets.
See `benchmarks.yaml` for dataset paths and time ranges (machine-readable).

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from aeon_qc import (
    heartbeat_gaps, heartbeat_duplicates,
    dropped_frames, epoch_gaps,
    run_qc, save_results, generate_report,
    diagnose_devices, schema_from_root,
)

## Configuration

Edit these values to point at the dataset you want to inspect. In the case that a bonsai crash occured, then continuing the experiment will produce a new data directory (a new epoch). In this case, your START and END times should cover both (or more) data epochs.

In [ ]:
ROOT = r"Z:\aeon\data\raw\AEON3\social0.2"
START = pd.Timestamp("2024-01-31T11:00:00", tz="UTC")
END   = pd.Timestamp("2024-02-01T21:00:00", tz="UTC")

diag = diagnose_devices(ROOT, start=START, end=END)

# Three-way presence table
all_devices = (
    (diag["registry"] or set())
    | (diag["metadata"] or set())
    | diag["filesystem"]
)
registry_col = f"registry ({diag['registry_key'] or 'no match'})"
rows = []
for device in sorted(all_devices):
    rows.append({
        "device": device,
        registry_col: "Y" if diag["registry"] and device in diag["registry"] else "",
        "metadata.yml": ("N/A" if diag["metadata"] is None
                         else ("Y" if device in diag["metadata"] else "")),
        "filesystem": "Y" if device in diag["filesystem"] else "",
    })
diag_df = pd.DataFrame(rows).set_index("device")
display(diag_df)

# Use filesystem scan as the working schema so downstream cells only
# attempt devices that actually have data on disk.
SCHEMA = schema_from_root(ROOT, start=START, end=END)

## Dataset timestamp bounds

Reports the first and last recorded timestamp for each device stream within the selected window.
Useful for confirming when data acquisition ended or detecting devices that stopped early. Note that in the first seconds as devices are initializing, Bonsai may produce Harp sync errors that are spurious. These are real, but data is only logged based on the first common heartbeats recieved, so will not be logged

In [ ]:
from swc.aeon.io import api as aeon_api
from aeon_qc.report import iter_readers

rows = []
for device_name, reader in iter_readers(SCHEMA):
    try:
        df = aeon_api.load(ROOT, reader, start=START, end=END)
        first_ts = df.index.min() if not df.empty else None
        last_ts  = df.index.max() if not df.empty else None
    except Exception:
        first_ts = last_ts = None
    rows.append({"device": device_name, "first_timestamp": first_ts, "last_timestamp": last_ts})

ts_df = pd.DataFrame(rows).set_index("device").sort_values("last_timestamp", ascending=False)
display(ts_df)

## Epoch gaps

Detects gaps between consecutive Bonsai session starts within the `START`/`END` window.
`load()` assembles `Metadata.yml` timestamps across all epoch directories in that range. Note that if metadata.yml is not present, that epoch will not be found here.

In [ ]:
epochs = epoch_gaps(ROOT, start=START, end=END)
n_gaps = int(epochs["gap_duration"].notna().sum())
print(f"{len(epochs)} epoch(s) in range, {n_gaps} inter-epoch gap(s)")
if not epochs.empty:
    if n_gaps > 0:
        real_gaps = epochs["gap_duration"].dropna()
        print(f"Shortest: {real_gaps.min()}, Longest: {real_gaps.max()}")
    display(epochs)

## Heartbeat gaps

Detects periods where a Harp device stops sending heartbeat events (~1 Hz).
A gap is flagged whenever the Harp `second` counter increments by more than 1 between
consecutive rows.

Returns a DataFrame indexed by gap-start time with `duration`, `n_missed`,
`second_before`, `second_after`, and `device` columns.

In [ ]:
from swc.aeon.io.reader import Heartbeat as HeartbeatReader
from aeon_qc.report import iter_readers

hb_devices = {name: reader for name, reader in iter_readers(SCHEMA)
              if isinstance(reader, HeartbeatReader)}

if not hb_devices:
    print("No Heartbeat devices found in this dataset")
else:
    print(f"Found {len(hb_devices)} heartbeat device(s): {list(hb_devices.keys())}\n")
    for device_name, hb_reader in hb_devices.items():
        gaps = heartbeat_gaps(ROOT, hb_reader, start=START, end=END)
        data_ok = gaps.attrs.get("data_found", True)
        if not data_ok:
            print(f"{device_name}: NO DATA on disk")
        elif gaps.empty:
            print(f"{device_name}: 0 gap(s)")
        else:
            print(f"{device_name}: {len(gaps)} gap(s)")
            display(gaps)

## Duplicate heartbeat seconds

Detects seconds where a Harp device emits more than one heartbeat with the same `second` counter value.
Duplicate heartbeats is a known issue that occurs when a device resychs to the clockSynchronizer broadcast. This momentarily inflates the `device_count` seen by Bonsai's `SynchronizerMonitor`, triggering a HarpSynch (expected devices) alert.

Each row is the first occurrence of a repeated `second`, with a `count` of how many heartbeats
were received for that second.

In [ ]:
if not hb_devices:
    print("No Heartbeat devices found in this dataset")
else:
    for device_name, hb_reader in hb_devices.items():
        dups = heartbeat_duplicates(ROOT, hb_reader, start=START, end=END)
        data_ok = dups.attrs.get("data_found", True)
        if not data_ok:
            print(f"{device_name}: NO DATA on disk")
        elif dups.empty:
            print(f"{device_name}: 0 duplicate second(s)")
        else:
            print(f"{device_name}: {len(dups)} duplicate second(s)")
            display(dups)

## Sync delta

The ClockSynchronizer broadcasts a clock signal (heartbeat) once per second to all Harp
devices, keeping them synchronised. All devices share a common logical counter (`second`);
aligning on that counter lets us compare Harp timestamps across devices and detect drift.

The ClockSynchronizer is used as the reference. Returns one row per second per
non-reference device with columns `second`, `device`, and `delta_seconds`.

In [ ]:
from aeon_qc import sync_delta
from aeon_qc.sync import MIN_DEVICES

if len(hb_devices) < MIN_DEVICES:
    print(f"sync_delta: need at least {MIN_DEVICES} heartbeat devices (found {len(hb_devices)})")
else:
    delta = sync_delta(ROOT, hb_devices, start=START, end=END)
    if delta.empty:
        print("sync_delta: no data")
    else:
        summary = (
            delta.groupby("device")["delta_seconds"]
            .agg(max_abs=lambda s: s.abs().max(), mean_abs=lambda s: s.abs().mean())
            .sort_values("max_abs", ascending=False)
        )
        print("Sync delta per device (seconds):")
        print(summary.to_string())

## Harp sync alerts

Parses `HarpSynch` alert entries written by the Bonsai `SynchronizerMonitor` into the
MessageLog. The Bonsai alert fires when **any** of three conditions hold:

- `DeviceCount != ExpectedDeviceCount` — a device has stopped sending heartbeats
- `MaxDifference > 0` — Harp clocks are misaligned by ≥1 second across devices
- `Abs(MeanUtcTimestamp − UtcNow) > 30 min` — the Harp network has drifted from wall-clock UTC

Returns one row per alert with `device_count`, `expected_device_count`, and `max_difference` columns.

In [ ]:
from aeon_qc import harp_sync_alerts
from swc.aeon.io.reader import Log as LogReader

log_readers = {name: reader for name, reader in iter_readers(SCHEMA)
               if isinstance(reader, LogReader)}

if not log_readers:
    print("No MessageLog readers found in schema")
else:
    for name, reader in log_readers.items():
        alerts = harp_sync_alerts(ROOT, reader, start=START, end=END)
        n_total = alerts.attrs.get("n_total_messages", 0)
        print(f"{name}: {n_total} total log entries, {len(alerts)} HarpSynch alert(s)")
        if not alerts.empty:
            display(alerts)

## Dropped frames

Detects dropped video frames via `hw_counter` jumps.
Returns a DataFrame indexed by the last received frame timestamp with `duration`,
`n_dropped`, `hw_counter_before`, `hw_counter_after`, and `device` columns.

In [ ]:
from swc.aeon.io.reader import Video as VideoReader

camera_devices = {name: reader for name, reader in iter_readers(SCHEMA)
                  if isinstance(reader, VideoReader)}

if not camera_devices:
    print("No camera devices found in this dataset")
else:
    print(f"Found {len(camera_devices)} camera(s): {list(camera_devices.keys())}\n")
    for device_name, cam_reader in camera_devices.items():
        drops = dropped_frames(ROOT, cam_reader, start=START, end=END)
        data_ok = drops.attrs.get("data_found", True)
        if not data_ok:
            print(f"{device_name}: NO DATA")
        elif drops.empty:
            print(f"{device_name}: 0 drop event(s)")
        else:
            total = drops["n_dropped"].sum()
            print(f"{device_name}: {len(drops)} drop event(s), {total} total frames dropped")
            display(drops)

## Encoder gaps

Detects dropped samples in the wheel encoder stream (~500 Hz, 2 ms interval).
Only gaps longer than the threshold (default 1 second) are flagged, ignoring sub-second jitter.
Returns a DataFrame indexed by gap-start time with `duration`, `n_missed`, and `device` columns.

In [ ]:
from aeon_qc import encoder_gaps
from aeon_qc.encoder import DEFAULT_THRESHOLD as ENCODER_THRESHOLD
from swc.aeon.io.reader import Encoder as EncoderReader

encoder_devices = {name: reader for name, reader in iter_readers(SCHEMA)
                   if isinstance(reader, EncoderReader)}

if not encoder_devices:
    print("No Encoder devices found in this dataset")
else:
    print(f"Found {len(encoder_devices)} encoder(s): {list(encoder_devices.keys())}")
    print(f"Gap threshold: {ENCODER_THRESHOLD}\n")
    for device_name, enc_reader in encoder_devices.items():
        gaps = encoder_gaps(ROOT, enc_reader, start=START, end=END)
        data_ok = gaps.attrs.get("data_found", True)
        if not data_ok:
            print(f"{device_name}: NO DATA")
        elif gaps.empty:
            print(f"{device_name}: 0 gap event(s)")
        else:
            total_missed = gaps["n_missed"].sum()
            print(f"{device_name}: {len(gaps)} gap event(s), {total_missed} total missed samples")
            display(gaps)

## Pellet delivery failures

Reports hardware-logged pellet delivery failures from the `MissedPellet` and
`RetriedDelivery` streams.
Returns a DataFrame indexed by event time with `outcome` (`missed` or `retried`)
and `device` columns.

In [ ]:
from aeon_qc import pellet_failures
from aeon_qc.report import iter_readers

feeder_devices = {}
for name, reader in iter_readers(SCHEMA):
    device, _, stream = name.partition(".")
    if stream in ("DeliverPellet", "MissedPellet", "RetriedDelivery"):
        feeder_devices.setdefault(device, {})[stream] = reader

feeder_devices = {d: s for d, s in feeder_devices.items() if "DeliverPellet" in s}

if not feeder_devices:
    print("No feeder devices found in schema")
else:
    for device_name, streams in feeder_devices.items():
        stats = pellet_failures(
            ROOT,
            deliver_reader=streams["DeliverPellet"],
            missed_reader=streams.get("MissedPellet"),
            retried_reader=streams.get("RetriedDelivery"),
            start=START, end=END,
        )
        n = stats.attrs.get("n_deliveries", 0)
        n_retried = stats.attrs.get("n_retried", 0)
        n_missed = stats.attrs.get("n_missed", 0)
        print(f"{device_name}: {n} deliveries, {n_retried} retried, {n_missed} missed")
        if not stats.empty:
            display(stats)

## Message log errors

Surfaces Warning and Error entries from the Bonsai message log.

In [ ]:
from aeon_qc import message_log_errors
from swc.aeon.io.reader import Log as LogReader

log_readers = {name: reader for name, reader in iter_readers(SCHEMA)
               if isinstance(reader, LogReader)}

if not log_readers:
    print("No MessageLog readers found in schema")
else:
    for name, reader in log_readers.items():
        errors = message_log_errors(ROOT, reader, start=START, end=END)
        n_total = errors.attrs.get("n_total", 0)
        print(f"{name}: {n_total} total log entries, {len(errors)} non-Info")
        if not errors.empty:
            display(errors)

## Environment state

Shows time spent in each environment state (e.g. `Running`, `Maintenance`) over
the selected time range.

In [ ]:
from aeon_qc import environment_state_durations
from swc.aeon.io.reader import Csv as CsvReader

env_state_readers = {name: reader for name, reader in iter_readers(SCHEMA)
                     if isinstance(reader, CsvReader) and "EnvironmentState" in name}

if not env_state_readers:
    print("No EnvironmentState readers found in schema")
else:
    for name, reader in env_state_readers.items():
        durations = environment_state_durations(ROOT, reader, start=START, end=END)
        if not durations.attrs.get("data_found", True):
            print(f"{name}: NO DATA")
        elif durations.empty:
            print(f"{name}: no state transitions found")
        else:
            totals = durations.groupby("state")["duration"].sum()
            print(f"{name}:")
            for state, total in totals.items():
                print(f"  {state}: {total}")

## Full scan

Runs all applicable QC checks across every stream in the schema at once.
Returns a dict of DataFrames keyed by device stream name. In addition to the
per-device metrics shown in the individual sections above, `run_qc` also
automatically includes `heartbeat_duplicates` (key: `<device>.duplicates`) and
`harp_sync_alerts` (key: `<device>.harp_sync_alerts`) for every Heartbeat and
MessageLog stream respectively.

In [ ]:
results = run_qc(ROOT, SCHEMA, start=START, end=END)

for name, df in results.items():
    found = df.attrs.get("data_found", True)
    status = "NO DATA" if not found else f"{len(df)} row(s)"
    print(f"{name}: {status}")

## Generate YAML report

Writes a structured YAML summary to disk with per-device `summary` and `detail` sections.
Also saves the full results dict as a pickle for downstream analysis.

In [ ]:
import os

os.makedirs("qc_results", exist_ok=True)

output = generate_report(ROOT, results, "qc_results/qc_report.yaml", start=START, end=END)
print(f"Report written to {output}")

pkl = save_results(results, "qc_results/qc_results.pkl")
print(f"Results pickled to {pkl}")

### Debug: inspect heartbeat data around a specific timestamp

In [ ]:
from swc.aeon.io import api as aeon_api

t = pd.Timestamp("2024-01-31T11:28:45", tz="UTC")
w = pd.Timedelta("10seconds")

frames = {}
for name, reader in hb_devices.items():
    df = aeon_api.load(ROOT, reader, start=t - w, end=t + w)
    if not df.empty:
        frames[name] = df["second"].astype(int).rename(name)

combined = pd.concat(frames.values(), axis=1, join="outer").sort_index()
display(combined)